In [1]:
!pip install librosa tensorflow pandas numpy matplotlib scikit-learn

In [81]:
import os
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

import joblib

In [94]:
dataset_path = r"C:\Users\edara\OneDrive\Desktop\HorizonTechX_EmotionRecognition\dataset\archive (9)"

In [95]:
def extract_features(file_path):

    try:
        audio, sample_rate = librosa.load(
            file_path,
            duration=3,
            offset=0.5
        )

        mfccs = librosa.feature.mfcc(
            y=audio,
            sr=sample_rate,
            n_mfcc=40
        )

        mfccs_scaled = np.mean(mfccs.T, axis=0)

        return mfccs_scaled

    except Exception as e:
        print("Error:", file_path)
        return None

In [96]:
emotion_map = {
    "01": "Neutral",
    "02": "Calm",
    "03": "Happy",
    "04": "Sad",
    "05": "Angry",
    "06": "Fearful",
    "07": "Disgust",
    "08": "Surprised"
}

In [97]:
features = []
emotions = []

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if file.endswith(".wav"):

            file_path = os.path.join(root, file)

            feature = extract_features(file_path)

            if feature is not None:

                emotion_code = file.split("-")[2]

                emotion = emotion_map.get(
                    emotion_code,
                    "Unknown"
                )

                features.append(feature)

                emotions.append(emotion)

print("Total Audio Files Loaded:", len(features))

Total Audio Files Loaded: 2880


In [102]:
X = np.array(features)

y = np.array(emotions)

print("X Shape:", X.shape)
print("Y Shape:", y.shape)

X Shape: (2880, 40)
Y Shape: (2880,)


In [104]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)

print("Classes:")
print(encoder.classes_)

Classes:
['Angry' 'Calm' 'Disgust' 'Fearful' 'Happy' 'Neutral' 'Sad' 'Surprised']


In [106]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 2304
Testing Samples: 576


In [108]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("Training Completed")

Training Completed


In [110]:
y_pred = model.predict(X_test)

In [112]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

Accuracy: 0.9270833333333334


In [114]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.92      0.90      0.91        79
           1       0.85      1.00      0.92        69
           2       0.91      0.93      0.92        84
           3       0.97      0.95      0.96        80
           4       0.95      0.95      0.95        82
           5       0.94      0.76      0.84        42
           6       0.96      0.90      0.93        61
           7       0.93      0.95      0.94        79

    accuracy                           0.93       576
   macro avg       0.93      0.92      0.92       576
weighted avg       0.93      0.93      0.93       576



In [116]:
joblib.dump(
    model,
    r"C:\Users\edara\OneDrive\Desktop\HorizonTechX_EmotionRecognition\model\emotion_model.pkl"
)

print("Model Saved Successfully")

Model Saved Successfully


In [118]:
saved_model = joblib.load(
    r"C:\Users\edara\OneDrive\Desktop\HorizonTechX_EmotionRecognition\model\emotion_model.pkl"
)

print("Model Loaded Successfully")

Model Loaded Successfully
